# Black-Scholes Pricing

## Data

Use the option-chain data in `data/option_data_AMZN.xlsx`.

The file contains AMZN options as of the quote date in the `market data` sheet. The option chains are listed equity options; for this exercise, treat them as European options and ignore dividends.

In [15]:
import sys
sys.path.insert(0, '../cmds')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

In [19]:
TICK = 'AMZN'
DATAFILE = 'option_data_AMZN.xlsx'

calls = pd.read_excel(DATAFILE, sheet_name='call chain')
puts = pd.read_excel(DATAFILE, sheet_name='put chain')
market = pd.read_excel(DATAFILE, sheet_name='market data').rename(columns={'Unnamed: 0': 'field'}).set_index('field')['data']

S = float(market.loc['equity price'])
rf = float(market.loc['Tbill'])
DATE = pd.to_datetime(market.loc['date'])
EXPRYDATE = pd.to_datetime(market.loc['option expiration'])
tau = to_maturity(expiration=EXPRYDATE, current_date=DATE)

# focal strike: nearest to spot among strikes quoted in BOTH chains
K = 250  # focal strike (quoted in both chains; see prose)
SIGMA = 0.30

NameError: name 'to_maturity' is not defined

In [7]:
market_frame = pd.DataFrame(market).T
market_frame[['equity price', 'Tbill']] = market_frame[['equity price', 'Tbill']].astype(float)
market_frame.style.format({'equity price': '${:,.2f}', 'Tbill': '{:.2%}'})

NameError: name 'market' is not defined

In [9]:
option_pair = pd.concat(
    {
        'call': calls.set_index('strike').loc[K, ['lastPrice', 'bid', 'ask', 'impliedVolatility', 'volume', 'openInterest']],
        'put': puts.set_index('strike').loc[K, ['lastPrice', 'bid', 'ask', 'impliedVolatility', 'volume', 'openInterest']],
    },
    axis=1,
).T

option_pair.style.format({
    'lastPrice': '${:,.2f}',
    'bid': '${:,.2f}',
    'ask': '${:,.2f}',
    'impliedVolatility': '{:.2%}',
    'volume': '{:,.0f}',
    'openInterest': '{:,.0f}',
})

NameError: name 'calls' is not defined

# 1. Option Chain

### 1.1.
For the expiration in the data file, plot the market quoted prices of the puts and calls across strikes.

Use `lastPrice` for the market option price.

### 1.2.
Make the same plot, but use quoted `impliedVolatility` rather than market price.

### 1.3.
What are the main differences between the price plot and the implied-volatility plot? What should you be careful about when interpreting the far out-of-the-money options?

***

# 2. Black-Scholes Values

Focus on the strike `K = 250`, with expiration equal to the option expiration in the data file.

### 2.1.
Using annualized volatility `SIGMA = 30%`, calculate the Black-Scholes value of the call and the put.

### 2.2.
Compare the Black-Scholes values to the listed `lastPrice` market quotes. Does `30%` look too high or too low relative to the market prices?

### 2.3.
Plot the Black-Scholes value of the call against the stock price, holding time-to-maturity, the risk-free rate, and volatility fixed.

Describe how the slope of the curve changes as the option moves from out-of-the-money to in-the-money.

***

# 3. Implied Volatility and Greeks

### 3.1.
Instead of assuming a volatility, use the market `lastPrice` to solve for the implied volatility of the call and the put at `K = 250`.

Compare your calculated implied volatilities to the quoted `impliedVolatility` values in the data.

### 3.2.
Using the implied volatility from the call, calculate the call's Delta, Gamma, Vega, and daily Theta.

### 3.3.
Reprice the call for volatilities from `20%` to `40%` in `5%` increments. How sensitive is the option value to volatility at this strike and maturity?